# <font color="blue">App: despliegue de la CNN con Gradio: dibuja un número y la CNN lo predice

En esta sección desplegamos el modelo convolucional entrenado en el **Cuaderno 8 A2** (guardado en Google Drive) como una **página web interactiva** usando [Gradio](https://www.gradio.app/).
La página tiene un lienzo (*Sketchpad*) donde se escribe un dígito **a mano alzada** con el mouse o el dedo, y la red
neuronal convolucional devuelve el dígito predicho junto con la probabilidad de cada clase.

Flujo de la predicción:

1. **Lienzo → escala de grises**: el trazo se convierte a una imagen en grises con el número en blanco sobre fondo negro (igual que MNIST).
2. **Recorte**: se recorta el rectángulo que contiene el trazo.
3. **Redimensión**: se escala para que el lado mayor mida 20 píxeles y se pega en un lienzo de 28x28 (así se construyó MNIST).
4. **Centrado por centro de masa**: se desplaza el dígito para que su centro de masa quede en el centro de la imagen.
5. **Normalización** a [0, 1] y forma `(1, 28, 28, 1)`.
6. **Predicción** con la CNN (`Conv2D` → `MaxPooling2D` → `Conv2D` → `MaxPooling2D` → `Dense` → `softmax`).

In [ ]:
!pip install -q gradio

### 1. Conectar Google Drive (funciona en Colab web y en VS Code con el kernel de Colab)

El modelo se lee de tu **Mi unidad**, dentro de la carpeta `Cuaderno8_CNN`. **Si la carpeta no existe, se crea.**

- **Colab en el navegador:** se monta Drive con `drive.mount()` como siempre.
- **VS Code con el kernel de Colab:** la extensión **no soporta `drive.mount()`**, así que se usa la **API de Google Drive** con una autorización OAuth. Solo hay que hacer esto **una vez**:
  1. Entra a [console.cloud.google.com](https://console.cloud.google.com/) y crea un proyecto (o usa uno existente).
  2. *APIs y servicios → Biblioteca* → busca **Google Drive API** → **Habilitar**.
  3. *APIs y servicios → Pantalla de consentimiento de OAuth* → tipo **Externo** → en *Usuarios de prueba* agrega tu correo de Gmail.
  4. *APIs y servicios → Credenciales → Crear credenciales → ID de cliente de OAuth* → tipo **App de escritorio** → copia el **ID de cliente** y el **Secreto del cliente**.

Al ejecutar la celda te pedirá el ID y el secreto (el secreto no queda guardado en el cuaderno). Luego te muestra un enlace: lo abres, eliges tu cuenta y aceptas. El navegador termina en una página `localhost` **que no carga, eso es normal**: copia la URL completa de la barra de direcciones y pégala en la caja que aparece en VS Code.

In [ ]:
import os
import shutil
from getpass import getpass
from urllib.parse import urlparse, parse_qs

NOMBRE_CARPETA = 'Cuaderno8_CNN'   # carpeta dentro de "Mi unidad" (se crea si no existe)
INTENTAR_MONTAR = True             # en VS Code drive.mount() no funciona: se pasa a la API automáticamente
DRIVE = {}


def _servicio_api_drive():
    # Autorización OAuth "copiar y pegar": funciona aunque el kernel esté en Colab y el navegador en tu PC
    from google_auth_oauthlib.flow import Flow
    from googleapiclient.discovery import build

    os.environ['OAUTHLIB_RELAX_TOKEN_SCOPE'] = '1'
    client_id = input('ID de cliente OAuth: ').strip()
    client_secret = getpass('Secreto del cliente OAuth: ').strip()
    config = {'installed': {
        'client_id': client_id,
        'client_secret': client_secret,
        'auth_uri': 'https://accounts.google.com/o/oauth2/auth',
        'token_uri': 'https://oauth2.googleapis.com/token',
        'redirect_uris': ['http://localhost'],
    }}
    flow = Flow.from_client_config(config, scopes=['https://www.googleapis.com/auth/drive'],
                                   redirect_uri='http://localhost')
    url, _ = flow.authorization_url(prompt='consent', access_type='offline')
    print('1) Abre este enlace, elige tu cuenta y acepta los permisos:\n')
    print(url)
    print('\n2) El navegador terminará en una página "localhost" que NO carga (es normal).')
    print('   Copia la URL completa de la barra de direcciones y pégala abajo.')
    respuesta = input('URL de localhost (o solo el código): ').strip()
    codigo = parse_qs(urlparse(respuesta).query).get('code', [respuesta])[0]
    flow.fetch_token(code=codigo)
    return build('drive', 'v3', credentials=flow.credentials)


def conectar_drive():
    # 1) Colab en el navegador: montar Drive
    if INTENTAR_MONTAR:
        try:
            from google.colab import drive
            if not os.path.isdir('/content/drive/MyDrive'):
                drive.mount('/content/drive')
            ruta = os.path.join('/content/drive/MyDrive', NOMBRE_CARPETA)
            if not os.path.isdir(ruta):
                os.makedirs(ruta)
                print('Carpeta creada en Mi unidad:', NOMBRE_CARPETA)
            DRIVE.update(modo='montado', carpeta=ruta)
            print('Google Drive montado. Carpeta:', ruta)
            return
        except Exception as e:
            print('No se pudo montar Drive (normal en VS Code):', e)
            print('Usando la API de Google Drive...\n')

    # 2) VS Code con kernel de Colab: API de Google Drive
    servicio = _servicio_api_drive()
    consulta = (f"name = '{NOMBRE_CARPETA}' and mimeType = 'application/vnd.google-apps.folder' "
                "and 'root' in parents and trashed = false")
    encontradas = servicio.files().list(q=consulta, fields='files(id)').execute()['files']
    if encontradas:
        carpeta_id = encontradas[0]['id']
        print('Carpeta encontrada en Mi unidad:', NOMBRE_CARPETA)
    else:
        carpeta_id = servicio.files().create(
            body={'name': NOMBRE_CARPETA, 'mimeType': 'application/vnd.google-apps.folder'},
            fields='id').execute()['id']
        print('Carpeta creada en Mi unidad:', NOMBRE_CARPETA)
    DRIVE.update(modo='api', servicio=servicio, carpeta_id=carpeta_id)


def _buscar_archivo_api(nombre):
    consulta = f"name = '{nombre}' and '{DRIVE['carpeta_id']}' in parents and trashed = false"
    archivos = DRIVE['servicio'].files().list(q=consulta, fields='files(id)').execute()['files']
    return archivos[0]['id'] if archivos else None


def subir_a_drive(ruta_local):
    # Copia un archivo local a Mi unidad/NOMBRE_CARPETA (si ya existe lo reemplaza)
    nombre = os.path.basename(ruta_local)
    if DRIVE['modo'] == 'montado':
        shutil.copy(ruta_local, os.path.join(DRIVE['carpeta'], nombre))
    else:
        from googleapiclient.http import MediaFileUpload
        media = MediaFileUpload(ruta_local, resumable=True)
        archivo_id = _buscar_archivo_api(nombre)
        if archivo_id:
            DRIVE['servicio'].files().update(fileId=archivo_id, media_body=media).execute()
        else:
            DRIVE['servicio'].files().create(body={'name': nombre, 'parents': [DRIVE['carpeta_id']]},
                                             media_body=media, fields='id').execute()
    print(f'Guardado en Mi unidad/{NOMBRE_CARPETA}/{nombre}')


def descargar_de_drive(nombre, destino='.'):
    # Trae un archivo de Mi unidad/NOMBRE_CARPETA y devuelve su ruta local
    ruta_local = os.path.join(destino, nombre)
    if DRIVE['modo'] == 'montado':
        ruta_drive = os.path.join(DRIVE['carpeta'], nombre)
        if not os.path.exists(ruta_drive):
            raise FileNotFoundError(f'No está {NOMBRE_CARPETA}/{nombre} en tu Drive. Ejecuta primero el cuaderno de entrenamiento.')
        shutil.copy(ruta_drive, ruta_local)
    else:
        from googleapiclient.http import MediaIoBaseDownload
        archivo_id = _buscar_archivo_api(nombre)
        if archivo_id is None:
            raise FileNotFoundError(f'No está {NOMBRE_CARPETA}/{nombre} en tu Drive. Ejecuta primero el cuaderno de entrenamiento.')
        with open(ruta_local, 'wb') as f:
            descarga = MediaIoBaseDownload(f, DRIVE['servicio'].files().get_media(fileId=archivo_id))
            terminado = False
            while not terminado:
                _, terminado = descarga.next_chunk()
    print(f'Descargado de Mi unidad/{NOMBRE_CARPETA}/{nombre}')
    return ruta_local


conectar_drive()

### 2. Cargar el modelo convolucional desde Google Drive

Se descarga `numerosC2.keras` de `Mi unidad/Cuaderno8_CNN/` (lo guardó el cuaderno de entrenamiento) y se carga con Keras.

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
import gradio as gr

ruta_modelo = descargar_de_drive('numerosC2.keras')
modelo_cnn = tf.keras.models.load_model(ruta_modelo)
clases = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
modelo_cnn.summary()

### 3. Preprocesamiento del dibujo (igual que MNIST)

MNIST tiene los dígitos **blancos sobre fondo negro**, centrados en una imagen de 28x28. El lienzo de Gradio entrega el trazo oscuro
sobre fondo blanco (o transparente), así que hay que invertirlo y centrarlo antes de pasarlo a la red; si no, la predicción falla mucho.

In [ ]:
def lienzo_a_gris(dibujo):
    """Convierte la salida del Sketchpad en una imagen en grises con trazo blanco y fondo negro (0-255)."""
    if isinstance(dibujo, dict):            # Gradio 4/5 entrega {'background', 'layers', 'composite'}
        dibujo = dibujo.get('composite')
    if dibujo is None:
        return None
    img = np.asarray(dibujo)
    if img.ndim == 3 and img.shape[2] == 4:
        alfa = img[..., 3].astype(np.float32)
        if alfa.min() < 250:                # fondo transparente: el trazo está en el canal alfa
            return alfa
        img = img[..., :3]
    if img.ndim == 3:
        img = img.mean(axis=2)
    gris = img.astype(np.float32)
    if gris.mean() > 127:                   # fondo blanco con trazo oscuro: invertir
        gris = 255.0 - gris
    return gris


def preprocesar_dibujo(dibujo):
    """Devuelve un tensor (1, 28, 28, 1) con valores en [0, 1], o None si el lienzo está vacío."""
    gris = lienzo_a_gris(dibujo)
    if gris is None:
        return None
    ys, xs = np.nonzero(gris > 30)
    if len(xs) == 0:
        return None

    # 1) Recortar el rectángulo que contiene el número
    recorte = gris[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    # 2) Escalar para que el lado mayor mida 20 px (conservando la proporción)
    alto, ancho = recorte.shape
    escala = 20.0 / max(alto, ancho)
    nuevo_ancho, nuevo_alto = max(1, round(ancho * escala)), max(1, round(alto * escala))
    recorte = cv2.resize(recorte, (nuevo_ancho, nuevo_alto), interpolation=cv2.INTER_AREA)

    # 3) Pegar en un lienzo negro de 28x28
    imagen = np.zeros((28, 28), dtype=np.float32)
    y0, x0 = (28 - nuevo_alto) // 2, (28 - nuevo_ancho) // 2
    imagen[y0:y0 + nuevo_alto, x0:x0 + nuevo_ancho] = recorte

    # 4) Centrar por centro de masa
    momentos = cv2.moments(imagen)
    if momentos['m00'] > 0:
        cx = momentos['m10'] / momentos['m00']
        cy = momentos['m01'] / momentos['m00']
        desplazamiento = np.float32([[1, 0, 14 - cx], [0, 1, 14 - cy]])
        imagen = cv2.warpAffine(imagen, desplazamiento, (28, 28))

    # 5) Normalizar a [0, 1] y añadir dimensiones de lote y canal
    imagen = imagen / max(imagen.max(), 1e-6)
    return imagen.reshape(1, 28, 28, 1)

### 4. Función de predicción

Recibe el dibujo, lo preprocesa, lo pasa por la CNN y devuelve:
- un diccionario `{dígito: probabilidad}` para el componente `gr.Label`, y
- la imagen de 28x28 que realmente "ve" la red (útil para entender qué le llega al modelo).

In [ ]:
def predecir(dibujo):
    entrada = preprocesar_dibujo(dibujo)
    if entrada is None:
        return {}, None
    probabilidades = modelo_cnn.predict(entrada, verbose=0)[0]
    resultado = {clases[i]: float(probabilidades[i]) for i in range(10)}
    vista_red = (entrada[0, :, :, 0] * 255).astype(np.uint8)
    vista_red = cv2.resize(vista_red, (140, 140), interpolation=cv2.INTER_NEAREST)
    return resultado, vista_red

### 5. Interfaz web y despliegue

- `gr.Sketchpad`: lienzo para escribir el número a mano alzada (pincel blanco grueso o negro según la versión; el preprocesamiento funciona con ambos).
- `gr.Label`: muestra el dígito predicho y las 3 clases más probables.
- `live=True`: predice automáticamente cada vez que se dibuja.
- `share=True`: genera un **enlace público** (`https://xxxx.gradio.live`) válido por 72 horas, que se puede abrir desde cualquier navegador o celular mientras el cuaderno siga ejecutándose.

In [ ]:
lienzo = gr.Sketchpad(
    label='Escribe un dígito (0-9)',
    type='numpy',
    image_mode='RGBA',
    canvas_size=(280, 280),
    brush=gr.Brush(default_size=18, colors=['#000000'], color_mode='fixed'),
)

demo = gr.Interface(
    fn=predecir,
    inputs=lienzo,
    outputs=[
        gr.Label(num_top_classes=3, label='Predicción de la CNN'),
        gr.Image(label='Lo que ve la red (28x28)', height=140, width=140),
    ],
    live=True,
    title='Reconocimiento de dígitos escritos a mano con una CNN',
    description='Dibuja un número del 0 al 9 en el lienzo. La red neuronal convolucional entrenada con MNIST predice qué dígito es.',
)

demo.launch(share=True, debug=True)

> **Nota:** `debug=True` mantiene la celda en ejecución mientras la página está en uso; para detenerla, interrumpe la celda (botón ⏹). Si quieres una página permanente (que no dependa de Colab), sube `app.py`, `requirements.txt` y `numerosC2.keras` (descárgalo de tu Drive) a un *Space* de [Hugging Face](https://huggingface.co/spaces) con SDK Gradio.